# Natural Language Processing

Natural Language Processing (NLP) is the field of helping computers work with human language. This notebook introduces **sentiment analysis**: predicting whether a restaurant review expresses a positive or negative opinion. It uses 1,000 labeled reviews from `data/Restaurant_Reviews.tsv`, pairing each review with its known sentiment.

This is a **supervised learning classification** task because the model learns from labeled examples. NLP can use either traditional machine learning or deep learning; this example uses **Bag of Words and Gaussian Naive Bayes**, without neural networks.

## What this notebook covers

1. **Load the dataset** with pandas.
2. **Clean the reviews** by removing non-letter characters, converting text to lowercase, removing common stopwords, and applying Porter stemming to reduce words to simpler forms. The word `not` is kept because negation can change sentiment.
3. **Convert text into numerical features** using `CountVectorizer` to build a Bag of Words representation with up to 1,500 features. Each review becomes a vector of word counts, without preserving word order.
4. **Split the data** into 80% training data and 20% test data using a fixed random seed.
5. **Train a Gaussian Naive Bayes classifier** to predict review sentiment.
6. **Evaluate predictions** using a confusion matrix and accuracy, comparing predicted labels with the known test labels.

By the end, you will understand a basic text-classification workflow: turning raw language into numerical features, learning from labeled examples, and measuring prediction quality.

In [5]:
import numpy as np
import pandas as pd

## Importing the dataset

In [6]:
# import the tab separated values (TSV) file, ignoring double quotes (can cause problems with apostrophes)
dataset = pd.read_csv('data/Restaurant_Reviews.tsv', delimiter ='\t', quoting = 3)

## Cleaning the texts

In [7]:
# import a regex and the Natural Language Toolkit (NLTK) library, which contains many useful NLP tools
import re
import nltk

# download stop words from NLTK, which are common words that are often removed from text analysis (not relevant words)
nltk.download('stopwords')

# import the stopwords and Porter stemmer (stemming contains only the root of the word to simplify the analysis)
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

ps = PorterStemmer()
all_stopwords = set(stopwords.words('english'))
# we want to keep the word "not" because it can change the sentiment of a review
all_stopwords.discard('not')
corpus = []
for i in range(0, dataset.shape[0]):
  # for each row:
  # remove punctuations and numbers, leaving only letters
  review = re.sub('[^a-zA-Z]', ' ', dataset['Review'][i])
  # convert to lowercase
  review = review.lower()
  # split into individual words in order to prepare for stemming and stopword removal
  review = review.split()
  # remove stopwords and apply stemming to each word
  review = [ps.stem(word) for word in review if not word in all_stopwords]
  # rejoin the words into a single string and add it to the corpus
  review = ' '.join(review)
  corpus.append(review)

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/costin/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [8]:
comparison = pd.DataFrame({
    'Original review': dataset['Review'].head(20).to_numpy(),
    'Processed corpus': corpus[:20]
})

from IPython.display import HTML, display
display(HTML(comparison.to_html(index=False)))

Original review,Processed corpus
Wow... Loved this place.,wow love place
Crust is not good.,crust not good
Not tasty and the texture was just nasty.,not tasti textur nasti
Stopped by during the late May bank holiday off Rick Steve recommendation and loved it.,stop late may bank holiday rick steve recommend love
The selection on the menu was great and so were the prices.,select menu great price
Now I am getting angry and I want my damn pho.,get angri want damn pho
Honeslty it didn't taste THAT fresh.),honeslti tast fresh
The potatoes were like rubber and you could tell they had been made up ahead of time being kept under a warmer.,potato like rubber could tell made ahead time kept warmer
The fries were great too.,fri great
A great touch.,great touch


## Creating the Bag of Words model

In [9]:
from sklearn.feature_extraction.text import CountVectorizer

# take only the most frequent words (up to 1,500) to reduce the number of features and avoid overfitting
# for example words like "stop", "late", "may", "bank", "holiday", "rick", "steve" etc in the reviews are not relevant to the sentiment, and they may have a low frequency, so they are not included in the Bag of Words
cv = CountVectorizer(max_features = 1500)
# make a 2D array of word counts (in this case we have 1,500 columns, one for each word, and 1,000 rows, one for each review)
X = cv.fit_transform(corpus).toarray()
# the last column of the dataset contains the sentiment labels (0 = negative, 1 = positive)
y = dataset.iloc[:, -1].values

## Splitting the dataset into the Training set and Test set

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state = 0)

## Training the Naive Bayes model on the Training set

In [11]:
from sklearn.naive_bayes import GaussianNB

classifier = GaussianNB()
classifier.fit(X_train, y_train)

,"priors priors: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
,"var_smoothing var_smoothing: float, default=1e-9Portion of the largest variance of all features that is added tovariances for calculation stability... versionadded:: 0.20",1e-09
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)number of training samples observed in each class.","ndarray[float64](2,)","[403.,397.]"
"class_prior_ class_prior_: ndarray of shape (n_classes,)probability of each class.","ndarray[float64](2,)","[0.5,0.5]"
"classes_ classes_: ndarray of shape (n_classes,)class labels known to the classifier.","ndarray[int64](2,)","[0,1]"
epsilon_ epsilon_: floatabsolute additive value to variances.,float64,1.15e-10
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,1500
"theta_ theta_: ndarray of shape (n_classes, n_features)mean of each feature per class.","ndarray[float64](2, 1500)","[[0.01,0. ,0. ,...,0. ,0. ,0.01], [0.01,0. ,0. ,...,0. ,0.01,0. ]]"
"var_ var_: ndarray of shape (n_classes, n_features)Variance of each feature per class... versionadded:: 1.0","ndarray[float64](2, 1500)","[[0.01,0. ,0. ,...,0. ,0. ,0.02], [0.01,0. ,0. ,...,0. ,0.01,0. ]]"


## Predicting the Test set results

In [12]:
y_pred = classifier.predict(X_test)
y_pred_reshaped = y_pred.reshape(len(y_pred), 1)
y_test_reshaped = y_test.reshape(len(y_test), 1)
print(np.concatenate((y_pred_reshaped, y_test_reshaped), 1))

[[1 0]
 [1 0]
 [1 0]
 [0 0]
 [0 0]
 [1 0]
 [1 1]
 [1 0]
 [1 0]
 [1 1]
 [1 1]
 [1 1]
 [1 0]
 [1 1]
 [1 1]
 [1 1]
 [0 0]
 [0 0]
 [0 0]
 [1 1]
 [0 0]
 [0 1]
 [1 1]
 [1 0]
 [1 0]
 [0 1]
 [1 1]
 [1 1]
 [1 1]
 [0 0]
 [1 1]
 [1 1]
 [1 1]
 [1 1]
 [1 1]
 [0 0]
 [1 0]
 [0 0]
 [1 0]
 [1 1]
 [1 1]
 [1 0]
 [1 1]
 [0 0]
 [0 0]
 [0 0]
 [1 0]
 [1 0]
 [0 0]
 [0 0]
 [1 1]
 [1 1]
 [1 1]
 [1 1]
 [1 0]
 [0 0]
 [1 1]
 [1 1]
 [0 0]
 [1 1]
 [1 0]
 [0 0]
 [1 0]
 [1 0]
 [1 1]
 [0 0]
 [1 1]
 [1 1]
 [1 1]
 [1 0]
 [1 1]
 [1 1]
 [1 1]
 [1 1]
 [0 0]
 [1 0]
 [1 1]
 [0 1]
 [0 0]
 [1 1]
 [0 0]
 [1 1]
 [1 1]
 [0 0]
 [1 1]
 [1 1]
 [1 0]
 [0 0]
 [1 1]
 [1 0]
 [0 0]
 [1 1]
 [0 0]
 [0 0]
 [1 0]
 [1 1]
 [1 0]
 [1 1]
 [1 1]
 [1 0]
 [0 1]
 [1 1]
 [1 1]
 [1 0]
 [0 1]
 [1 0]
 [1 1]
 [1 1]
 [0 0]
 [0 1]
 [0 1]
 [1 1]
 [0 0]
 [1 0]
 [1 1]
 [0 0]
 [1 1]
 [1 1]
 [1 1]
 [1 1]
 [1 1]
 [0 0]
 [1 1]
 [1 0]
 [0 0]
 [0 0]
 [1 1]
 [1 0]
 [0 0]
 [1 1]
 [1 0]
 [1 1]
 [0 0]
 [0 0]
 [1 1]
 [1 1]
 [1 1]
 [1 1]
 [1 1]
 [1 0]
 [0 1]
 [1 1]
 [1 1]

## Making the Confusion Matrix

In [13]:
from sklearn.metrics import confusion_matrix, accuracy_score

cm = confusion_matrix(y_test, y_pred)
print(cm)
accuracy_score(y_test, y_pred)

[[55 42]
 [12 91]]


0.73